## Assignment 1:

- Using Numpy to implement the soft-margin SVM model. 
- Train this model using SGD method on the [Chest X-Ray Images (Pneumonia)](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia). Resize the images to $128 \times 128$.
- Evaluate this model using Precision, Recall, and F1 metrics.

In [21]:
import numpy as np
from tqdm import tqdm

class SVM:
    def __init__(self, C: float, epochs: int = 1000, lr: float = 1e-4):
        self.C = C
        self.epochs = epochs
        self.lr = lr

        self.W = None
        self.b = None
        self.losses = []

    def fit(self, X: np.ndarray, y: np.ndarray):
        n_samples, n_features = X.shape

        y = np.where(y <= 0, -1, 1)

        self.W = np.zeros(n_features)
        self.b = 0

        pbar = tqdm(range(self.epochs))

        for _ in pbar:
            for i in range(n_samples):
                condition = y[i] * (np.dot(self.W, X[i]) + self.b) >= 1

                if condition:
                    self.W -= self.lr * (2 * self.W)
                else:
                    self.W -= self.lr * (2 * self.W - self.C * y[i] * X[i])
                    self.b -= self.lr * (-self.C * y[i])

            y_hat = self.predict(X)
            loss = self.hinge_loss(y, y_hat)
            self.losses.append(loss)

            pbar.set_postfix({
                "loss": loss
            })

    def predict(self, X: np.ndarray):
        if len(X.shape) == 1:
            X = np.expand_dims(X, axis=0)

        y_hat = self.W.T @ X.T + self.b
        return y_hat

    def hinge_loss(self, y: np.ndarray, y_hat: np.ndarray):
        delta = 1 - y * y_hat
        return 0.5 * (self.W.T @ self.W) + self.C * np.maximum(0, delta).sum()

In [22]:
import cv2
import os

def load_images(folder):
    data = []
    labels = []

    for label in ["NORMAL", "PNEUMONIA"]:
        path = os.path.join(folder, label)
        class_label = 0 if label == "NORMAL" else 1

        for img_name in os.listdir(path):
            img_path = os.path.join(path, img_name)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            img = cv2.resize(img, (128, 128))
            img = img.flatten() / 255.0  

            data.append(img)
            labels.append(class_label)

    return np.array(data), np.array(labels)

In [23]:
from sklearn.metrics import precision_score, recall_score, f1_score

def evaluate(y_true, y_pred):
    y_pred = np.where(y_pred == -1, 0, 1)

    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    print("Precision:", precision)
    print("Recall:", recall)
    print("F1-score:", f1)

In [24]:
X_train, y_train = load_images("data/chest_xray/train")
X_test, y_test = load_images("data/chest_xray/test")

svm = SVM(C=1.0, epochs=100, lr=0.0001)
svm.fit(X_train, y_train)

preds = svm.predict(X_test)

evaluate(y_test, preds)

100%|██████████| 100/100 [01:08<00:00,  1.47it/s, loss=3.95e+3]

Precision: 0.625
Recall: 1.0
F1-score: 0.7692307692307693


## Assigment 2:
- Implement SVM method using a machine learning library (such as sklearn or sktorch).
- Train this model on the [Chest X-Ray Images (Pneumonia)](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia). Resize the images to $128 \times 128$.
- Evaluate this model using Precision, Recall, and F1 metrics.
- Compare the results of SVM using library to those of implemented SVM.

In [25]:
from sklearn.svm import SVC

model = SVC(kernel='linear', C=1.0)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

evaluate(y_test, y_pred)

Precision: 0.625
Recall: 1.0
F1-score: 0.7692307692307693
